# Solution PromoPro — étape 2A : évaluation d'un modèle déjà entraîné

Modèle testé : `Yytsi/floorplan-to-3d-walls` (U-Net, encodeur ResNet-34, 4 classes : sol / mur / porte / fenêtre), entraîné sur **CubiCasa5K**.

⚠️ **Licence** : le modèle est publié sous MIT, mais CubiCasa5K est **CC BY-NC 4.0** (usage non commercial). Ce carnet sert **uniquement à évaluer** la qualité atteignable ; ce modèle ne peut pas être mis en production dans PromoPro sans licence commerciale CubiCasa. Voir IA-INTERNE.md.

Ce carnet est prévu pour **Google Colab** (niveau gratuit, CPU suffisant pour une inférence). Exécutez les cellules dans l'ordre. À la fin, vous téléchargez le masque et un JSON au même format que celui rendu par Gemini, pour comparer.

In [ ]:
# 1. Dépendances (2 à 3 minutes)
%pip -q install segmentation-models-pytorch huggingface_hub safetensors opencv-python-headless pyyaml

In [ ]:
# 2. Téléchargement des poids et de la configuration
from huggingface_hub import hf_hub_download
import yaml, json
poids = hf_hub_download('Yytsi/floorplan-to-3d-walls', 'best.safetensors')
config = yaml.safe_load(open(hf_hub_download('Yytsi/floorplan-to-3d-walls', 'config.yaml')))
print(json.dumps(config, indent=2)[:3000])  # à lire : nom de l'encodeur, nombre de classes, taille d'image, normalisation

In [ ]:
# 3. Construction du modèle et chargement des poids
import torch, segmentation_models_pytorch as smp
from safetensors.torch import load_file

def lire(cle, defaut):
    # la configuration publiée peut nommer les champs différemment : on cherche la clé à tous les niveaux
    pile = [config]
    while pile:
        d = pile.pop()
        if isinstance(d, dict):
            if cle in d: return d[cle]
            pile.extend(d.values())
    return defaut

encodeur = lire('encoder_name', lire('encoder', 'resnet34'))
classes = int(lire('classes', lire('num_classes', 4)))
taille = int(lire('image_size', lire('img_size', 512)))
print('encodeur', encodeur, '| classes', classes, '| taille', taille)

modele = smp.Unet(encoder_name=encodeur, encoder_weights=None, classes=classes)
etat = load_file(poids)
manquants, inattendus = modele.load_state_dict(etat, strict=False)
print('clés manquantes :', len(manquants), '| inattendues :', len(inattendus))
if manquants or inattendus:
    print('→ Si ces listes ne sont pas vides, le préfixe des clés diffère : afficher list(etat)[:5] et adapter (ex. retirer "model.").')
    print(list(etat)[:5])
modele.eval()

In [ ]:
# 4. Déposez l'image du plan (villa_plan_2d.png) quand la boîte de dialogue s'ouvre
from google.colab import files
depose = files.upload()
chemin = next(iter(depose))
print('plan :', chemin)

In [ ]:
# 5. Inférence : masque des 4 classes (0 sol, 1 mur, 2 porte, 3 fenêtre)
import cv2, numpy as np
img = cv2.cvtColor(cv2.imread(chemin), cv2.COLOR_BGR2RGB)
H, W = img.shape[:2]
cote = (taille // 32) * 32
redim = cv2.resize(img, (cote, cote), interpolation=cv2.INTER_AREA).astype(np.float32) / 255.0
moy, ecart = np.array([0.485, 0.456, 0.406]), np.array([0.229, 0.224, 0.225])  # normalisation ImageNet (à vérifier dans config)
x = torch.from_numpy(((redim - moy) / ecart).transpose(2, 0, 1)).float()[None]
with torch.no_grad():
    logits = modele(x)
masque = logits.argmax(1)[0].numpy().astype(np.uint8)
masque = cv2.resize(masque, (W, H), interpolation=cv2.INTER_NEAREST)
print('pixels par classe :', {c: int((masque == c).sum()) for c in range(classes)})

couleurs = np.array([[255, 255, 255], [20, 20, 20], [230, 120, 0], [0, 120, 230]], dtype=np.uint8)
superposition = (0.55 * img + 0.45 * couleurs[np.clip(masque, 0, 3)]).astype(np.uint8)
cv2.imwrite('masque.png', cv2.cvtColor(superposition, cv2.COLOR_RGB2BGR))
from IPython.display import Image, display
display(Image('masque.png', width=700))

In [ ]:
# 6. Pièces = composantes connexes du sol séparées par les murs ; portes = composantes de la classe porte
# Résultat au même format que le JSON Gemini (coordonnées relatives), sans nom de pièce : ce modèle ne lit pas le texte.
sol = (masque == 0).astype(np.uint8)
sol = cv2.morphologyEx(sol, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
n, etiquettes, stats, _ = cv2.connectedComponentsWithStats(sol, connectivity=4)
seuil = 0.004 * H * W  # ignore les miettes (< 0,4 % de l'image)
pieces = []
for i in range(1, n):
    x0, y0, w, h, aire = stats[i]
    if aire < seuil or w >= 0.98 * W:  # le fond extérieur touche tout le bord
        continue
    pieces.append({'nom': f'Pièce {len(pieces) + 1}', 'x': round(x0 / W, 3), 'y': round(y0 / H, 3), 'largeur': round(w / W, 3), 'hauteur': round(h / H, 3)})
porte = (masque == 2).astype(np.uint8)
n2, _, stats2, centres = cv2.connectedComponentsWithStats(porte, connectivity=8)
portes = [{'x': round(centres[i][0] / W, 3), 'y': round(centres[i][1] / H, 3), 'mur': 'vertical' if stats2[i][3] > stats2[i][2] else 'horizontal'} for i in range(1, n2) if stats2[i][4] > 0.0002 * H * W]
resultat = {'pieces': pieces, 'portes': portes, 'remarques': 'modèle Yytsi/floorplan-to-3d-walls (évaluation, licence CubiCasa5K non commerciale)'}
print(json.dumps(resultat, ensure_ascii=False, indent=2))
json.dump(resultat, open('villa_plan_2d.modele.json', 'w'), ensure_ascii=False, indent=2)
files.download('villa_plan_2d.modele.json'); files.download('masque.png')

## 7. Comparer avec Gemini

Déposez `villa_plan_2d.gemini.json` (rendu par `scripts/prototypes/test-lecture-plan.ts`) et exécutez la cellule suivante : pour chaque pièce Gemini, la pièce du modèle qui la recouvre le mieux (IoU des rectangles) est indiquée. Un IoU ≥ 0,7 signifie une pièce bien retrouvée.

In [ ]:
gem = json.load(open(next(iter(files.upload()))))
def iou(a, b):
    ax1, ay1, ax2, ay2 = a['x'], a['y'], a['x'] + a['largeur'], a['y'] + a['hauteur']
    bx1, by1, bx2, by2 = b['x'], b['y'], b['x'] + b['largeur'], b['y'] + b['hauteur']
    inter = max(0, min(ax2, bx2) - max(ax1, bx1)) * max(0, min(ay2, by2) - max(ay1, by1))
    union = a['largeur'] * a['hauteur'] + b['largeur'] * b['hauteur'] - inter
    return inter / union if union else 0
print(f"{'pièce Gemini':<18}{'meilleure pièce modèle':<24}{'IoU':>6}")
for p in gem['pieces']:
    meilleur = max(pieces, key=lambda q: iou(p, q), default=None)
    print(f"{p['nom']:<18}{(meilleur['nom'] if meilleur else '—'):<24}{(iou(p, meilleur) if meilleur else 0):>6.2f}")
print(f"\nportes : Gemini {len(gem['portes'])}, modèle {len(portes)}")